In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.0+cpu
CUDA available: False


In [2]:
from pathlib import Path
from miditok import REMI

TOKENIZER_PATH = Path("../configs/tokenizer.json")

tokenizer = REMI(params=TOKENIZER_PATH)

print("Tokenizer loaded")
print("Vocabulary size:", len(tokenizer))

Tokenizer loaded
Vocabulary size: 20000


c:\Users\aqdas\Minor_Project\piano-music-composition\.venv\Lib\site-packages\miditok\classes.py:886: UserWarning: Argument nb_tempos has been renamed num_tempos, you should consider to updateyour code with this new argument name.
  return cls(**input_dict, **kwargs)


In [3]:
from transformers import AutoModelForCausalLM

MODEL_NAME = "NathanFradet/Maestro-REMI-bpe20k"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype="auto"
).to(device)


print("Model loaded.")
print("Device:", device)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model loaded.
Device: cpu


In [4]:
from pathlib import Path

MAESTRO_DIR = Path("../data/raw/maestro")

midi_files = list(MAESTRO_DIR.rglob("*.midi")) + list(MAESTRO_DIR.rglob("*.mid"))

print("MIDI files found:", len(midi_files))
print("Example:", midi_files[524])

MIDI files found: 1276
Example: ..\data\raw\maestro\2011\MIDI-Unprocessed_02_R1_2011_MID--AUDIO_R1-D1_08_Track08_wav.midi


In [5]:
midi_path = midi_files[1]

tokens = tokenizer(midi_path)

print(tokens)

[TokSequence(tokens=['Bar_None', 'Position_17', 'Pitch_43', 'Velocity_47', 'Duration_0.1.8', 'Pitch_31', 'Velocity_39', 'Duration_0.1.8', 'Position_18', 'Pitch_44', 'Velocity_47', 'Duration_0.1.8', 'Pitch_32', 'Velocity_39', 'Duration_0.1.8', 'Position_19', 'Pitch_46', 'Velocity_55', 'Duration_0.2.8', 'Pitch_34', 'Velocity_39', 'Duration_0.1.8', 'Position_21', 'Pitch_48', 'Velocity_55', 'Duration_1.1.4', 'Pitch_36', 'Velocity_39', 'Duration_1.1.4', 'Bar_None', 'Position_16', 'Pitch_53', 'Velocity_55', 'Duration_3.1.2', 'Pitch_62', 'Velocity_71', 'Duration_4.0.1', 'Pitch_56', 'Velocity_63', 'Duration_3.1.2', 'Pitch_48', 'Velocity_47', 'Duration_4.0.1', 'Bar_None', 'Position_22', 'Pitch_54', 'Velocity_47', 'Duration_0.3.8', 'Pitch_63', 'Velocity_55', 'Duration_0.4.8', 'Position_23', 'Pitch_59', 'Velocity_47', 'Duration_0.1.8', 'Pitch_44', 'Velocity_47', 'Duration_0.4.8', 'Position_27', 'Pitch_60', 'Velocity_55', 'Duration_1.0.4', 'Pitch_39', 'Velocity_47', 'Duration_0.5.8', 'Pitch_48', '

In [6]:
# Convert the token sequence to integer IDs
token_ids = tokens[0].ids

print("Number of tokens:", len(token_ids))
print("First 50 token IDs:")
print(token_ids[:50])

Number of tokens: 1913
First 50 token IDs:
[1482, 738, 3027, 148, 821, 2432, 149, 1220, 1896, 2411, 248, 21, 258, 1097, 38, 867, 47, 1066, 41, 946, 33, 812, 2845, 1440, 1352, 8790, 29, 192, 157, 2310, 24, 200, 15228, 1978, 2693, 2260, 1836, 1298, 1520, 674, 3497, 5868, 155, 1085, 156, 2629, 19814, 3988, 158, 1030]


In [7]:
print("Model vocabulary size:", model.config.vocab_size)
print("Tokenizer vocabulary size:", len(tokenizer))

Model vocabulary size: 20000
Tokenizer vocabulary size: 20000


In [8]:
print(max(token_ids))

19996


In [10]:
# print("n_positions:", model.config.n_positions)
# print("max_position_embeddings:", model.config.max_position_embeddings)

# print("Input length:", input_ids.shape[1])
# print("Requested total:", input_ids.shape[1] + 1024)

In [11]:
CONTEXT_LENGTH = 500

In [12]:
len(token_ids)

1913

In [13]:
import torch



input_ids = torch.tensor(
    [token_ids[:CONTEXT_LENGTH]],
    dtype=torch.long,
    device=device
)

prompt_length = input_ids.shape[1]

print("Input shape:", input_ids.shape)

with torch.no_grad():
    output = model.generate(
        input_ids=input_ids,
        max_new_tokens=512,
        do_sample=True,
        temperature=1.0,
        top_k=15,
        top_p=0.95,
    )

print("Generated shape:", output.shape)
print("Generated tokens:", output.shape[1])

Input shape: torch.Size([1, 500])
Generated shape: torch.Size([1, 1012])
Generated tokens: 1012


In [14]:
# Remove the original prompt, keeping only the generated continuation

prompt_length = input_ids.shape[1]
generated_ids = output[0, prompt_length:].tolist()

print("Generated continuation tokens:", len(generated_ids))
print(generated_ids[:50])

Generated continuation tokens: 512
[3820, 159, 53, 1662, 5821, 159, 2810, 194, 6152, 142, 6152, 140, 6969, 139, 15001, 159, 5699, 194, 4773, 246, 1534, 154, 2608, 3593, 276, 2621, 256, 246, 723, 141, 8012, 140, 15580, 158, 5699, 735, 33, 556, 146, 1534, 211, 1500, 3829, 360, 246, 1534, 144, 982, 140, 40]


In [15]:
from miditok import TokSequence

generated_sequence = TokSequence(
    ids=generated_ids,
    are_ids_encoded=True
)

print("TokSequence created.")

TokSequence created.


In [16]:
# Decode the generated BPE IDs back to MIDI
generated_midi = tokenizer.decode([generated_ids])

print("MIDI decoded successfully.")

MIDI decoded successfully.


In [17]:
output_path = "../outputs/pretrained/generated_pretrained.mid"
generated_midi.dump_midi(output_path)

print("Saved:", output_path)

Saved: ../outputs/pretrained/generated_pretrained.mid


In [18]:
import torch

# Get the model's BOS token
bos_token_id = model.config.bos_token_id

print("BOS token:", bos_token_id)

# Start with only BOS — no MIDI prompt
input_ids = torch.tensor([[bos_token_id]], dtype=torch.long, device=device)

with torch.no_grad():
    output = model.generate(
        input_ids=input_ids,
        max_new_tokens=2000,
        do_sample=True,
        temperature=2.5,
        top_k=15,
        top_p=0.95,
        pad_token_id=model.config.eos_token_id,
    )

generated_ids = output[0].tolist()

print("Generated tokens:", len(generated_ids))

BOS token: 2
Generated tokens: 2001


In [19]:
generated_midi = tokenizer.decode([generated_ids])

output_path = "../outputs/pretrained/unprompted.mid"
generated_midi.dump_midi(output_path)

print("Saved:", output_path)

Saved: ../outputs/pretrained/unprompted.mid


In [20]:
print("n_positions:", model.config.n_positions)
print("max_position_embeddings:", model.config.max_position_embeddings)

print("Input length:", input_ids.shape[1])
print("Requested total:", input_ids.shape[1] + 1024)

n_positions: 2048
max_position_embeddings: 2048
Input length: 1
Requested total: 1025
